# 활성화 함수: GELU, SwiGLU 등 - 현대 활성화 함수 비교

- Tutorial ID: `adv-8-1`
- Tutorial: 활성화 함수: GELU, SwiGLU 등
- Section ID: `adv-8-1-1`
- Section: 현대 활성화 함수 비교


## 이 노트북에서 배우는 것

이 노트북은 신경망(neural network)에서 사용하는 **활성화 함수(activation function)**
를 아주 기초적인 질문부터 시작해서 차근차근 살펴봅니다.

다루는 순서는 다음과 같습니다.

1. 활성화 함수가 왜 필요한가? (선형 변환만으로는 안 되는 이유)
2. ReLU — 가장 단순하고 널리 쓰여온 활성화 함수, 그리고 그 한계
3. GELU — BERT, GPT 등 트랜스포머 계열에서 쓰는 부드러운(smooth) 활성화 함수
4. SiLU(Swish) — "자기 자신을 게이트로 쓰는" 활성화 함수
5. 기울기(gradient)로 "죽은 뉴런" 문제를 직접 눈으로 확인하기
6. 표와 그래프로 세 함수를 한눈에 비교하기
7. 게이팅(gating)이라는 개념
8. SwiGLU — LLaMA 등 최신 LLM이 사용하는, 게이팅을 적용한 구조
9. 희소성(sparsity) — 활성화 함수에 따라 얼마나 많은 출력이 0에 가까워지는가

**사전 지식**: 파이썬 기초 문법과 numpy 배열(array)에 익숙하면 충분합니다.
신경망의 순전파(forward pass, 입력을 받아 출력을 계산해 나가는 과정)가
"행렬 곱(matmul) + 활성화 함수"의 반복이라는 정도만 알고 있어도 괜찮습니다.

> 이 노트북은 코드를 한 번 실행하고 끝내는 용도가 아닙니다.
> 각 함수가 "왜" 그런 모양을 가지는지, 입력값이나 하이퍼파라미터를 바꿔보면서
> 직접 느껴보는 데 목적이 있습니다. 코드를 실행한 뒤 숫자를 바꿔서
> 다시 실행해 보는 것을 권장합니다.


In [ ]:
# ============================================================
# 코드 읽는 법 — 현대 활성화 함수 비교
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

### 들어가기 전에: 사용할 라이브러리

이 노트북에서는 두 가지 라이브러리만 사용합니다.

- `numpy`: 배열(array) 연산을 위한 라이브러리입니다. 벡터/행렬 곱셈, 지수함수(exp),
  tanh 등을 제공합니다. 신경망의 계산은 대부분 numpy의 배열 연산만으로도 충분히
  표현할 수 있습니다.
- `matplotlib`: 함수의 모양을 그래프로 그려서 눈으로 직접 확인하기 위한 라이브러리입니다.

(참고: torch 같은 딥러닝 프레임워크가 없어도, 활성화 함수의 핵심 동작은
numpy만으로 충분히 이해할 수 있습니다.)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("활성화 함수 비교")
print("=" * 60)

## 1. 활성화 함수는 왜 필요할까요?

신경망의 한 층(layer)은 보통 다음과 같은 두 단계로 계산됩니다.

1. **선형 변환(linear transformation)**: 입력에 가중치 행렬을 곱한다
   → `입력 @ 가중치`
2. **활성화 함수(activation function)**: 그 결과에 비선형 함수를 적용한다
   → `활성화함수(입력 @ 가중치)`

그렇다면 2번 단계, 즉 활성화 함수를 빼고 `입력 @ 가중치` 계산만 여러 층 쌓으면
어떻게 될까요? 바로 아래 코드 셀에서 직접 확인해 보겠습니다.

**먼저 결론을 말하면:** 활성화 함수 없이 선형 변환(행렬 곱)만 아무리 여러 번
쌓아도, 전체는 결국 "단 하나의 선형 변환"과 수학적으로 똑같아집니다.
즉 층을 100개 쌓으나 1개를 쌓으나 표현할 수 있는 함수의 종류(표현력)가
같아져 버립니다. 활성화 함수는 이 층과 층 사이에 비선형성(non-linearity),
즉 "구부러짐"을 만들어 주는 역할을 하며, 이 덕분에 신경망이 직선/평면으로는
표현할 수 없는 복잡한 패턴을 배울 수 있습니다.


In [ ]:
# ------------------------------------------------------------
# 실험: 활성화 함수 없이 선형 레이어만 두 번 쌓으면 무슨 일이 생길까?
# ------------------------------------------------------------
np.random.seed(0)   # 같은 결과를 재현하기 위해 난수 시드를 고정 (없으면 실행할 때마다 값이 달라짐)

x = np.array([1.0, 2.0, 3.0])          # 입력 벡터 (3차원이라고 가정)

W1 = np.random.randn(3, 4)             # 1번째 레이어 가중치: 3차원 -> 4차원으로 변환
W2 = np.random.randn(4, 2)             # 2번째 레이어 가중치: 4차원 -> 2차원으로 변환

# (A) 레이어를 두 번 "따로" 통과시키는 경우 (활성화 함수 없이)
h = x @ W1                  # 1단계 통과: shape (3,) -> (4,)
out_two_layers = h @ W2     # 2단계 통과: shape (4,) -> (2,)

# (B) 두 가중치 행렬을 미리 하나로 곱쳐 두고, "한 번에" 통과시키는 경우
W_combined = W1 @ W2        # (3,4) @ (4,2) -> (3,2) 짜리 행렬 하나로 합쳐짐
out_one_layer = x @ W_combined

print("두 레이어를 따로 통과한 결과 :", out_two_layers)
print("하나로 합친 레이어 결과     :", out_one_layer)
print("두 결과가 (거의) 같은가?    :", np.allclose(out_two_layers, out_one_layer))
print()
print("=> 활성화 함수가 없으면, 레이어를 아무리 쌓아도")
print("   결국 '행렬 하나'를 곱한 것과 수학적으로 동일해집니다.")
print("   이러면 층을 깊게 쌓는 의미가 없습니다 — 그래서 활성화 함수가 꼭 필요합니다.")

## 2. ReLU (Rectified Linear Unit)

가장 단순하고, 가장 널리 쓰여온 활성화 함수입니다.

**수식**: `ReLU(x) = max(0, x)`

말로 풀면: "입력이 양수면 그대로 두고, 음수면 0으로 만든다" 입니다.

**장점**
- 계산이 매우 단순하고 빠릅니다 (0과 비교만 하면 됨).
- 기울기(gradient)가 1 아니면 0이라서 계산이 간단합니다.

**단점: 죽은 뉴런(Dying ReLU) 문제**
- 입력이 한 번 음수가 되면 출력은 항상 0이고, 그 지점에서의 기울기도 0입니다.
- 학습(역전파, backpropagation)은 이 기울기를 이용해 가중치를 업데이트하는데,
  기울기가 0이면 "이 방향으로 가중치를 바꿔도 출력이 안 바뀐다"는 뜻이라
  더 이상 학습이 일어나지 않습니다. 이런 뉴런을 "죽었다(dead)"고 표현합니다.

아래에서 직접 숫자를 넣어 확인해 봅시다.


In [ ]:
def relu(x):
    """
    ReLU (Rectified Linear Unit)
    입력이 0보다 크면 그대로, 0보다 작거나 같으면 0을 반환합니다.
    """
    return np.maximum(0, x)


# 간단한 숫자로 직접 동작을 확인해 봅니다
print("ReLU 동작 확인")
print(f"  relu(-3.0) = {relu(-3.0)}   (음수 -> 0)")
print(f"  relu( 0.0) = {relu(0.0)}   (0은 그대로 0)")
print(f"  relu( 2.5) = {relu(2.5)}   (양수 -> 그대로 통과)")

## 3. GELU (Gaussian Error Linear Unit)

BERT, GPT-2/3, ViT 등 많은 트랜스포머(Transformer) 모델에서 사용해 온
활성화 함수입니다 (Hendrycks & Gimpel, 2016년 제안). ReLU의 "뚝 끊기는" 모양
대신, 부드럽게(smooth) 휘어지는 모양을 가집니다.

**핵심 아이디어**

ReLU는 "입력이 양수인지 음수인지"만 보고 통과(1) 또는 차단(0)을 결정합니다
(이분법적). 반대로 GELU는 입력값 x를 표준정규분포(standard normal distribution)
위에 놓고, "이 값이 얼마나 큰 값인지"를 확률적으로 따져서 통과 비율을 정합니다.

수식으로 쓰면 다음과 같습니다.

```
GELU(x) = x * Φ(x)
```

여기서 Φ(x)는 표준정규분포의 누적분포함수(CDF, x보다 작은 값이 나올 확률)입니다.
x가 클수록 Φ(x)는 1에 가까워지고(거의 다 통과), x가 작을수록(많이 음수일수록)
Φ(x)는 0에 가까워집니다(거의 차단). 이 Φ(x)가 바로 "얼마나 통과시킬지" 정하는
게이트(gate) 역할을 합니다.

다만 Φ(x)를 정확히 계산하려면 적분이 필요해서 매번 계산하기엔 비용이 큽니다.
그래서 실제 구현(BERT, GPT-2 등)에서는 tanh를 이용한 근사식을 사용하며,
아래 코드의 수식이 바로 그 근사식입니다.

```
GELU(x) ≈ 0.5 * x * (1 + tanh( sqrt(2/π) * (x + 0.044715 * x^3) ))
```

복잡해 보이지만, 괄호 안의 `tanh(...)` 부분이 -1 ~ 1 사이의 값을 만들고,
거기에 1을 더해서 2로 나누면(즉 0.5를 곱하면) 0~1 사이의 "통과 비율"이
만들어지는 구조입니다. ReLU의 `max(0,x)`를 부드럽게 만든 버전이라고
생각하면 됩니다.


In [ ]:
def gelu(x):
    """
    GELU (Gaussian Error Linear Unit) - tanh 근사식

    원래 식: GELU(x) = x * Φ(x)  (Φ = 표준정규분포 누적분포함수)
    아래는 BERT/GPT-2 등에서 실제로 쓰는 tanh 근사 버전입니다.
    """
    gate_ratio = 0.5 * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))
    # gate_ratio: "입력을 얼마나 통과시킬지"를 나타내는 0~1 사이의 비율
    return x * gate_ratio


# ReLU와 비교하며 동작을 확인합니다 (괄호 안은 통과 비율)
print("GELU 동작 확인")
for x in [-3.0, -1.0, 0.0, 1.0, 3.0]:
    ratio = 0.5 * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))
    print(f"  x={x:>5.1f}  ->  gelu(x)={gelu(x):>7.4f}   (통과 비율 {ratio:.1%})")

print()
print("ReLU와 다른 점: x가 약간 음수(-1.0)여도 출력이 정확히 0이 아니라")
print("              작은 음수 값을 가집니다. 즉, 정보가 완전히 막히지 않습니다.")

## 4. SiLU (Sigmoid Linear Unit), 일명 Swish

SiLU는 LLaMA 계열 등 최근 모델에서 자주 쓰입니다 (정확히는 바로 뒤에서 다룰
SwiGLU의 재료로 사용됩니다). "Swish"라는 이름으로 Google 연구진이 2017년
발표한 논문(Searching for Activation Functions)에서 널리 알려졌습니다.

**수식**: `SiLU(x) = x * sigmoid(x)`

GELU와 아이디어가 비슷합니다 — "입력 x에, 0~1 사이의 통과 비율을 곱한다" —
인데, 그 통과 비율을 정규분포 CDF 대신 **시그모이드(sigmoid) 함수**로
정한다는 점이 다릅니다.

시그모이드 함수는 입력이 무엇이든 0~1 사이로 눌러주는 함수입니다.

```
sigmoid(x) = 1 / (1 + e^(-x))
```

x가 클수록 sigmoid(x)는 1에 가까워지고, x가 작을수록(음수로 갈수록) 0에
가까워집니다. SiLU는 "자기 자신을 보고, 자기 자신을 얼마나 통과시킬지
스스로 결정한다"는 의미에서 **self-gating(자기 게이팅)**이라고도 부릅니다
(입력값 자체가 게이트 역할도 동시에 하기 때문입니다).


In [ ]:
def sigmoid(x):
    """
    시그모이드 함수: 어떤 실수 입력이든 0~1 사이의 값으로 변환합니다.
    np.clip으로 x를 -500~500 사이로 제한하는 이유는, x가 아주 큰 음수일 때
    np.exp(-x)가 너무 커져서 오버플로(overflow) 경고가 나는 것을 막기 위함입니다.
    """
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


def silu(x):
    """
    SiLU (Sigmoid Linear Unit), 일명 Swish
    수식: SiLU(x) = x * sigmoid(x)
    """
    return x * sigmoid(x)


# 동작 확인 (괄호 안은 sigmoid 값 = 통과 비율)
print("SiLU 동작 확인")
for x in [-3.0, -1.0, 0.0, 1.0, 3.0]:
    print(f"  x={x:>5.1f}  ->  silu(x)={silu(x):>7.4f}   (sigmoid(x)={sigmoid(x):.4f})")

## 5. "죽은 뉴런" 문제를 기울기로 직접 확인하기

기울기(gradient)는 "입력을 아주 살짝 바꿨을 때 출력이 얼마나 바뀌는가"를
나타냅니다. 신경망 학습은 이 기울기를 따라 가중치를 조금씩 업데이트하는
과정이므로, 기울기가 0이면 그 가중치는 더 이상 업데이트되지 않습니다.

아래에서는 미분 공식을 손으로 유도하는 대신, **수치 미분(numerical
differentiation)** 이라는 방법으로 기울기를 근사해서 확인합니다.
입력값을 아주 약간(`eps`) 늘렸을 때와 줄였을 때의 출력 차이를 이용하는
방법입니다.


In [ ]:
def numerical_gradient(fn, x, eps=1e-5):
    """
    함수 fn의 x에서의 기울기를 수치적으로 근사합니다.
    f'(x) ≈ ( f(x+eps) - f(x-eps) ) / (2 * eps)
    """
    return (fn(x + eps) - fn(x - eps)) / (2 * eps)


print("입력이 음수일 때, 각 함수의 기울기(gradient) 비교")
print(f"{'x':>6} {'ReLU 기울기':>14} {'GELU 기울기':>14} {'SiLU 기울기':>14}")
print("-" * 52)
for x in [-3.0, -2.0, -1.0, -0.5]:
    g_relu = numerical_gradient(relu, x)
    g_gelu = numerical_gradient(gelu, x)
    g_silu = numerical_gradient(silu, x)
    print(f"{x:>6.1f} {g_relu:>14.4f} {g_gelu:>14.4f} {g_silu:>14.4f}")

print()
print("ReLU는 음수 구간에서 기울기가 항상 정확히 0입니다 -> 학습 신호가 완전히 끊김 (죽은 뉴런)")
print("GELU/SiLU는 음수 구간에서도 0이 아닌 기울기를 가집니다 -> 학습이 계속 이어질 수 있음")

## 6. 표와 그래프로 한눈에 비교하기

지금까지 본 세 함수(ReLU, GELU, SiLU)를 표로, 그리고 그래프로 비교해
보겠습니다. 숫자로 보는 것보다 그래프로 보면 "부드러움의 차이"가
훨씬 잘 느껴집니다.


In [ ]:
x_vals = np.array([-3, -2, -1, -0.5, 0, 0.5, 1, 2, 3])

print(f"{'x':>6} {'ReLU':>8} {'GELU':>8} {'SiLU':>8}")
print("-" * 35)
for x in x_vals:
    print(f"{x:>6.1f} {relu(x):>8.4f} {gelu(x):>8.4f} {silu(x):>8.4f}")

print()
print("관찰 포인트")
print("  - x가 2 이상이 되면 세 함수가 거의 비슷해집니다 (셋 다 x를 거의 그대로 통과).")
print("  - x < 0 구간에서 ReLU만 정확히 0이고, GELU/SiLU는 작은 음수값을 가집니다.")
print("  - GELU와 SiLU는 모양이 서로 매우 비슷합니다 (둘 다 부드러운 곡선).")

In [ ]:
# 그래프로 직접 비교해 봅니다.
# (그래프 안의 글자는 한글 폰트가 없는 환경에서도 깨지지 않도록 영어로 표기합니다.
#  실제 화면에 □□□ 같은 깨진 글자가 보인다면, 이것이 바로 '한글 폰트 미설치' 문제입니다.)

x_range = np.linspace(-5, 5, 400)   # -5부터 5까지 400개의 촘촘한 점을 만들어 매끄러운 곡선을 그림

plt.figure(figsize=(9, 5.5))
plt.plot(x_range, relu(x_range), label="ReLU", linewidth=2)
plt.plot(x_range, gelu(x_range), label="GELU", linewidth=2)
plt.plot(x_range, silu(x_range), label="SiLU (Swish)", linewidth=2)

plt.axhline(y=0, color="gray", linewidth=0.8)   # y=0 기준선
plt.axvline(x=0, color="gray", linewidth=0.8)   # x=0 기준선
plt.xlabel("input x")
plt.ylabel("output f(x)")
plt.title("Activation Functions: ReLU vs GELU vs SiLU")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("그래프에서 확인할 점:")
print("  - ReLU는 x=0에서 '뾰족하게' 꺾입니다 (미분이 불가능한 지점).")
print("  - GELU/SiLU는 x=0 근처에서 매끄럽게(smooth) 휘어집니다.")
print("  - GELU/SiLU는 x가 0보다 약간 작을 때 아주 살짝 0보다 '아래로' 내려갔다가")
print("    다시 올라오는 모양을 가집니다 (작은 음의 영역). ReLU에는 없는 특징입니다.")

## 7. 게이팅(Gating)이란?

SwiGLU를 이해하려면 먼저 "게이트(gate)"라는 개념을 짚고 넘어가야 합니다.

게이트는 한마디로 **"정보를 얼마나 통과시킬지를 정하는 또 다른 신호"**
입니다. 수도꼭지의 밸브를 떠올리면 쉽습니다 — 물(정보)이 흐르고 있어도,
밸브(게이트)를 얼마나 열어두느냐에 따라 실제로 흘러나오는 물의 양이
달라집니다.

사실 위에서 본 GELU/SiLU의 "통과 비율(0~1)"도 게이트의 한 종류였습니다 —
다만 그 게이트가 입력 자기 자신으로부터 계산되었을 뿐입니다. 아래
아주 단순한 예제로 게이팅의 감을 잡아 봅시다.


In [ ]:
# 아주 단순화한 게이팅 예시
info = np.array([10.0, 20.0, 30.0])     # 전달하고 싶은 정보(값)
gate = np.array([0.0, 0.5, 1.0])        # 각 정보를 얼마나 통과시킬지 (0=완전 차단, 1=완전 통과)

print("정보값       :", info)
print("게이트 값    :", gate, " (0=차단, 1=완전통과)")
print("통과된 결과  :", info * gate)   # 같은 위치(원소)끼리 곱함 (element-wise 곱)
print()
print("=> 같은 정보라도 게이트 값에 따라 얼마나 살아남을지가 달라집니다.")
print("   SwiGLU에서는 이 '게이트 값'을 SiLU를 이용해 입력으로부터 직접 계산합니다.")

## 8. GLU와 SwiGLU 구조

**GLU (Gated Linear Unit)** 의 기본 아이디어는 다음과 같습니다.

> 입력 x를 두 가지 다른 방식으로 각각 한 번씩 선형 변환(투영, projection)해서,
> 하나는 "내용물(value)"로 쓰고 다른 하나는 "게이트(gate)"로 쓴다.

수식으로 쓰면:

```
GLU(x) = (x @ W_value) * 게이트함수(x @ W_gate)
```

여기서 게이트함수 자리에 무엇을 쓰느냐에 따라 이름이 달라집니다
(Noam Shazeer, "GLU Variants Improve Transformer", 2020년 논문에서 정리).

- 게이트함수로 sigmoid를 쓰면 → GLU
- 게이트함수로 ReLU를 쓰면 → ReGLU
- 게이트함수로 GELU를 쓰면 → GEGLU
- 게이트함수로 **SiLU(Swish)** 를 쓰면 → **SwiGLU** (LLaMA, Mistral, PaLM 등에서 사용)

즉 SwiGLU는 "SiLU를 게이트로 사용하는 GLU"라는 뜻입니다. 앞에서 만든
`silu` 함수를 그대로 재사용해서 SwiGLU를 만들 수 있습니다.

**실제 트랜스포머의 FFN(Feed-Forward Network)에서는 다음 3개의 가중치를 사용합니다.**

| 가중치 | 역할 | shape |
|---|---|---|
| `W_gate` | 입력을 "게이트 신호"로 투영 | (d_model, d_ff) |
| `W_value` | 입력을 "실제 내용물"로 투영 | (d_model, d_ff) |
| `W_out` | 게이트로 걸러진 결과를 다시 원래 차원으로 투영 | (d_ff, d_model) |

여기서:
- `d_model` (코드에서는 `d`): 모델이 기본적으로 사용하는 차원 (입출력 벡터의 크기)
- `d_ff`: FFN 내부에서 일시적으로 "확장"하는 차원. 보통 `d_model`보다 큽니다
  (더 넓은 공간에서 다양한 패턴을 표현하기 위함입니다). 예를 들어 LLaMA 계열은
  보통 `d_ff`를 `d_model`의 약 8/3(≈2.7)배 정도로 둡니다 — SwiGLU는 가중치
  행렬을 3개(W_gate, W_value, W_out) 쓰기 때문에, 행렬을 2개만 쓰는 일반
  FFN(보통 d_ff = 4 × d_model)과 전체 파라미터 수를 비슷하게 맞추려고
  이렇게 비율을 줄여서 사용합니다.

아래 코드에서는 각 단계마다 shape(배열의 모양)을 출력해서, 차원이 어떻게
바뀌는지 직접 추적해 봅니다.


In [ ]:
print("=" * 60)
print("SwiGLU (LLaMA 스타일 FFN) 단계별 추적")
print("=" * 60)

np.random.seed(42)   # 재현 가능성을 위해 난수 시드를 고정합니다 (같은 코드 -> 같은 결과)

d = 8        # d_model: 입력/출력 벡터의 차원
d_ff = 16    # 내부에서 확장하는 차원 (보통 d_model보다 큼)

# 세 개의 가중치 행렬을 무작위로 초기화합니다.
# (실제 학습에서는 이 값들이 훈련 과정을 통해 점점 더 좋은 값으로 조정됩니다)
W_gate  = np.random.randn(d, d_ff) * 0.1   # shape: (8, 16)
W_value = np.random.randn(d, d_ff) * 0.1   # shape: (8, 16)
W_out   = np.random.randn(d_ff, d) * 0.1   # shape: (16, 8)

x = np.random.randn(d)   # 입력 벡터 하나, shape: (8,)
print(f"[입력]         x.shape              = {x.shape}")

# 1단계: 입력을 게이트용으로 투영한 뒤 SiLU를 적용 -> "얼마나 통과시킬지" 계산
gate_raw = x @ W_gate          # shape: (8,) @ (8,16) -> (16,)
gate = silu(gate_raw)          # SiLU를 적용해 통과 비율로 변환, shape 유지: (16,)
print(f"[게이트 투영]   (x @ W_gate).shape    = {gate_raw.shape}")
print(f"[게이트 활성화] silu(...).shape       = {gate.shape}")

# 2단계: 입력을 내용물(value)용으로 투영 -> 활성화 함수 없이 그대로 사용
value = x @ W_value            # shape: (8,) @ (8,16) -> (16,)
print(f"[값 투영]      (x @ W_value).shape    = {value.shape}")

# 3단계: 게이트와 값을 원소별로 곱함 (element-wise multiply, 같은 shape끼리 곱함)
gated = gate * value           # shape: (16,) * (16,) -> (16,) (각 위치끼리 곱함)
print(f"[게이트 적용]   (gate * value).shape  = {gated.shape}")

# 4단계: 다시 원래 차원(d)으로 투영해서 출력
out = gated @ W_out            # shape: (16,) @ (16,8) -> (8,)
print(f"[출력 투영]    (gated @ W_out).shape  = {out.shape}")

print()
print(f"요약: 입력 dim {d} -> (내부에서 {d_ff}로 확장) -> 다시 출력 dim {out.shape[0]}")
print(f"게이트가 0.1보다 크게 열린 비율: {np.mean(np.abs(gate) > 0.1):.1%}")
print()
print("핵심 정리:")
print("  - W_gate, W_value 두 개의 서로 다른 투영이 같은 입력 x로부터 동시에 이루어집니다.")
print("  - 하나(gate)는 SiLU를 거쳐 '문지기' 역할을 하고,")
print("  - 다른 하나(value)는 그대로 '내용물' 역할을 합니다.")
print("  - 두 결과를 원소별로 곱해서, 게이트가 닫힌 위치의 정보는 줄어들고")
print("    게이트가 열린 위치의 정보만 살아남아 다음 레이어로 전달됩니다.")

### SwiGLU를 여러 입력(토큰 시퀀스)에 한 번에 적용하기

실제 트랜스포머에서는 벡터 하나가 아니라, "토큰 여러 개로 이루어진 문장
(시퀀스)"을 한꺼번에 처리합니다. 다행히 위에서 작성한 코드는 입력이
1차원 벡터든 2차원 행렬(토큰 여러 개)이든 **거의 그대로** 동작합니다.
numpy의 행렬 곱(`@`)이 자동으로 차원을 맞춰주기 때문입니다.

아래에서 "토큰 5개로 이루어진 문장"을 가정하고, 위에서 만든 것과 동일한
가중치(W_gate, W_value, W_out)로 SwiGLU를 적용해 봅시다.


In [ ]:
# 이번에는 벡터 1개가 아니라, "토큰이 5개인 문장"을 가정해 봅니다.
seq_len = 5   # 문장 안의 토큰(단어) 개수

X_seq = np.random.randn(seq_len, d)   # shape: (5, 8) -> 토큰 5개, 각 토큰은 8차원 벡터
print(f"[입력 시퀀스]  X_seq.shape = {X_seq.shape}   (토큰 {seq_len}개 x 차원 {d})")

# 가중치(W_gate, W_value, W_out)는 바로 위 셀에서 만든 것을 그대로 재사용합니다.
gate_seq  = silu(X_seq @ W_gate)            # (5,8) @ (8,16) -> (5,16)
value_seq = X_seq @ W_value                 # (5,8) @ (8,16) -> (5,16)
out_seq   = (gate_seq * value_seq) @ W_out  # (5,16) @ (16,8) -> (5,8)

print(f"[게이트]      gate_seq.shape  = {gate_seq.shape}")
print(f"[값]          value_seq.shape = {value_seq.shape}")
print(f"[최종 출력]    out_seq.shape   = {out_seq.shape}")
print()
print("=> 토큰 하나짜리 벡터에서 했던 것과 완전히 같은 연산인데,")
print("   입력의 첫 번째 차원만 '토큰 개수'만큼 늘어났을 뿐입니다.")
print("   실제 트랜스포머의 FFN도 (batch, seq_len, d_model) 형태의")
print("   3차원 텐서에 대해 똑같은 원리로 동작합니다.")

## 9. 희소 활성화(Sparse Activation) 비교

마지막으로, 각 활성화 함수가 "출력을 얼마나 0에 가깝게(=사실상 차단)
만드는지"를 더 큰 규모(1000차원, 100개 샘플)에서 비교해 봅니다.

여기서 "희소(sparse)하다"는 것은 출력 벡터의 많은 원소가 0이거나 0에
매우 가까워서, 사실상 다음 레이어로 전달되는 정보가 적다는 뜻입니다.
이는 두 가지 의미를 가질 수 있습니다.

- (장점이 될 수 있음) 일부 뉴런만 활성화되므로 계산을 절약하거나, 뉴런들이
  서로 다른 역할로 "전문화(specialize)"되는 데 도움이 될 수 있습니다.
- (단점이 될 수 있음) 너무 많이 죽으면 모델의 표현력이 줄어들고, 학습이
  느려지거나 멈출 수 있습니다 (위에서 본 죽은 뉴런 문제).

ReLU와 GELU/SiLU는 "0에 가까워지는 방식" 자체가 다르다는 점에 주의하면서
비교해 봅시다.


In [ ]:
print("희소 활성화 분석 (1000차원 벡터 100개)")
print("-" * 50)

np.random.seed(0)
X = np.random.randn(100, 1000)   # 평균 0, 표준편차 1인 정규분포를 따르는 난수 100x1000개

threshold = 0.01   # 절댓값이 이 값보다 작으면 "사실상 비활성화(거의 0)"로 간주

for name, fn in [("ReLU", relu), ("GELU", gelu), ("SiLU", silu)]:
    acts = fn(X)
    near_zero_pct = np.mean(np.abs(acts) < threshold)
    print(f"  {name}: 절댓값이 {threshold} 미만인 비율 = {near_zero_pct:.1%}")

print()
print("[참고] ReLU는 음수 입력을 '근사치가 아니라 정확히' 0으로 만듭니다.")
exact_zero_relu = np.mean(relu(X) == 0)
print(f"        ReLU 출력이 정확히 0인 비율: {exact_zero_relu:.1%}")
print("        (표준정규분포는 양수/음수 비율이 약 50:50이므로, 이 값도 약 50%에 가깝습니다)")
print()
print("        반면 GELU/SiLU는 입력이 0에 아주 가까울 때만 출력도 0에 가까워질 뿐,")
print("        '정확히 0'이 되는 입력은 사실상 x=0 하나뿐입니다.")
print("        즉 GELU/SiLU의 '비활성화'는 ReLU처럼 완전한 차단이 아니라,")
print("        '아주 약하게만 통과시킴'에 더 가깝습니다.")

In [ ]:
# 막대 그래프로도 비교해 봅니다.
names = ["ReLU", "GELU", "SiLU"]
near_zero_ratios = [np.mean(np.abs(fn(X)) < threshold) for fn in [relu, gelu, silu]]

plt.figure(figsize=(6, 4.5))
plt.bar(names, near_zero_ratios, color=["#4C72B0", "#DD8452", "#55A868"])
plt.ylabel("ratio of near-zero outputs")
plt.title(f"Near-zero output ratio (threshold = {threshold})")
plt.ylim(0, 1)
for i, v in enumerate(near_zero_ratios):
    plt.text(i, v + 0.02, f"{v:.1%}", ha="center")   # 막대 위에 퍼센트 숫자를 직접 표시
plt.grid(True, axis="y", alpha=0.3)
plt.show()

## 10. 전체 정리

| 함수 | 수식(핵심) | 특징 | 주로 쓰이는 곳 |
|---|---|---|---|
| ReLU | `max(0, x)` | 단순/빠름, 음수에서 기울기 0 (죽은 뉴런 위험) | 고전적인 CNN 등 |
| GELU | `x * Φ(x)` (정규분포 CDF 기반, tanh로 근사) | 부드러움, 음수에서도 기울기 존재 | BERT, GPT-2/3, ViT |
| SiLU(Swish) | `x * sigmoid(x)` | GELU와 비슷하게 부드러움, self-gating | SwiGLU의 재료로 사용 |
| SwiGLU | `(silu(x@W_gate) * (x@W_value)) @ W_out` | 입력을 게이트/값 두 갈래로 나눠 처리 (GLU 계열) | LLaMA, PaLM, Mistral 등 최신 LLM |

**한 줄 요약**

- ReLU는 단순하지만 음수를 "완전히, 영구히" 차단할 위험이 있습니다.
- GELU/SiLU는 ReLU를 부드럽게 만들어 이 문제를 줄였습니다.
- SwiGLU는 한 발 더 나아가, "통과시킬지 말지"를 입력 자체로부터 학습된
  게이트로 동적으로 결정하게 만든 구조이며, 현재 많은 최신 LLM의 기본
  선택지가 되었습니다.

**다음 단계로 직접 해볼 수 있는 것**

- `d`, `d_ff` 값을 바꿔서 SwiGLU 코드를 다시 실행해보기
- `np.random.seed()` 값을 바꿔서 통계치(희소 비율 등)가 얼마나 달라지는지 확인해보기
- `threshold` 값을 0.001, 0.1 등으로 바꿔가며 희소 비율이 어떻게 변하는지 관찰해보기
- `seq_len`을 늘려서(예: 50) SwiGLU 시퀀스 적용 코드를 다시 실행해보기
